In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/moleculenet-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "support-vector-machines.py"

# 以支持向量預測致突變性與溶解度

首次執行會下載 Maom Lab 的 MutagenLou2023（CC BY 4.0）及 MoleculeNet ESOL。可設定 DEEPCHEM_DATA_DIR 重用原始檔。要做練習，請在最後一格前新增以 `%%writefile -a "{script}"` 開頭的程式格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from sklearn.metrics import mean_absolute_error

SEED = 42
# ESOL: 1,128 筆水溶解度 log(mol/L)，DOI 10.1021/ci034243x。
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path.home() / ".cache" / "deepchem"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

def read_splits(loader):
    # 原始檔可共用，每次建立的特徵快取則彼此獨立。
    with tempfile.TemporaryDirectory() as scratch:
        tasks, splits, _ = loader(
            featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
            transformers=[], data_dir=str(cache), save_dir=scratch, reload=True)
        assert len(tasks) == 1 and all(np.all(s.w[:, 0] > 0) for s in splits)
        result = [(np.array([fingerprint(str(smi)) for smi in s.ids]),
                   s.y[:, 0].copy(), s.ids.copy()) for s in splits]
    print(tasks[0], "train / valid / test:", *(len(y) for _, y, _ in result))
    return result

from rdkit.Chem import MACCSkeys

def fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(smiles)
    return np.asarray(MACCSkeys.GenMACCSKeys(mol), dtype=float)


In [ ]:
%%writefile -a "{script}"
from sklearn.svm import SVC, SVR
from sklearn.metrics import balanced_accuracy_score

# MutagenLou2023 (Maom Lab mirror, CC BY 4.0; Lou et al., 2023).
# Original rows/labels preserved in the cached files. Only structure enters X.
from pathlib import Path
from urllib.request import urlopen
import csv
import hashlib
import io
import os
import tempfile
import numpy as np
from rdkit import Chem
from sklearn.model_selection import train_test_split

def mutagen_splits():
    revision = "a9780a60774eb522b78bd92b83c30d8deae00ff5"
    source = f"https://huggingface.co/datasets/maomlab/MutagenLou2023/resolve/{revision}/train_test"
    base = Path(os.environ.get("DEEPCHEM_DATA_DIR", tempfile.gettempdir())) / "mutagenlou-a9780a6"
    base.mkdir(parents=True, exist_ok=True)
    expected = {"train": "ad0d4e2850afdc07c130c3679cc8ae2ad9f7c80999158325d8de986c39f950e1",
                "test": "08786199fbbdb6db614d5f65a5abdfcd26d882f0cc844bbeb4ccfa241127b394"}
    groups = {}
    for name, digest in expected.items():
        file = base / (name + ".csv")
        raw = file.read_bytes() if file.exists() else urlopen(f"{source}/{name}.csv", timeout=90).read()
        if hashlib.sha256(raw).hexdigest() != digest:
            raise ValueError(f"MutagenLou2023 checksum mismatch: {name}")
        if not file.exists():
            file.write_bytes(raw)
        labels = {}
        for row in csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))):
            mol = Chem.MolFromSmiles(row["SMILES"])
            if mol is not None:
                y = int(row["Y"])
                assert y in (0, 1)
                labels.setdefault(Chem.MolToSmiles(mol), set()).add(y)
        groups[name] = labels
    # Preserve mirrored test membership: no test structure may enter development.
    dev = [(s, next(iter(v))) for s,v in groups["train"].items() if len(v)==1 and s not in groups["test"]]
    test = [(s, next(iter(v))) for s,v in groups["test"].items() if len(v)==1]
    fit, valid = train_test_split(dev, test_size=0.15, random_state=42, stratify=[y for _,y in dev])
    return [(np.array([s for s,_ in rows]), np.array([y for _,y in rows])) for rows in (fit,valid,test)]

mutagen = [(np.array([fingerprint(s) for s in smiles]), labels, smiles)
           for smiles, labels in mutagen_splits()]
print("MutagenLou2023 train / valid / test:", *(len(y) for _, y, _ in mutagen))
esol = read_splits(dc.molnet.load_delaney)
candidates = [("linear", {"kernel": "linear", "C": 1}),
              ("RBF C=1", {"kernel": "rbf", "C": 1, "gamma": "scale"}),
              ("RBF C=10", {"kernel": "rbf", "C": 10, "gamma": "scale"})]
fig, axes = plt.subplots(1, 2, figsize=(8, 3.3), layout="constrained")
for ax, splits, estimator, metric, maximize, title in [
    (axes[0], mutagen, SVC, balanced_accuracy_score, True, "Ames mutagenicity"),
    (axes[1], esol, SVR, mean_absolute_error, False, "ESOL regression")]:
    (xt, yt, _), (xv, yv, _), (xs, ys, _) = splits
    models, scores = [], []
    for name, params in candidates:
        model = estimator(**params).fit(xt, yt)
        score = metric(yv, model.predict(xv))
        print(title, name, "valid", round(score, 4), "supports", len(model.support_))
        models.append(model)
        scores.append(score)
    winner = int(np.argmax(scores) if maximize else np.argmin(scores))
    print("selected", candidates[winner][0], "test", round(metric(ys, models[winner].predict(xs)), 4))
    ax.bar([c[0] for c in candidates], scores,
           color=["#146c74" if i == winner else "#a7b7b9" for i in range(len(scores))])
    ax.set(title=title, ylabel="Validation balanced accuracy" if maximize else "Validation MAE (log S)")
    ax.set_ylim(0, 1 if maximize else max(scores) * 1.15)
fig.savefig(figure_dir / "09-08-support-vector-machines.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-08-support-vector-machines.png")))